# Offline Reinforcement Learning Training
This notebook performs offline training of reinforcement learning agents on Adroit manipulation tasks. 
It uses datasets previously saved in `.npz` format and evaluates various algorithms implemented in d3rlpy. 
Training logs and policies are saved for each task and algorithm combination.

## Import Required Libraries and Check Versions
We begin by importing all the required libraries and verifying library versions and GPU availability.

In [ ]:
import minari
import d3rlpy
import numpy as np
import os
import torch
import logging
from datetime import datetime
import gc

# Version checking
print(f"d3rlpy version: {d3rlpy.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Task and Algorithm Setup
Select which tasks and algorithms will be used for training.

In [ ]:
# Select tasks and algorithms to evaluate
tasks = ['relocate', 'door', 'hammer', 'pen']
algorithms = ['iql', 'td3bc', 'awac', 'cql', 'bc']

# Type of dataset to load (e.g., 'expert', 'medium')
dataset_type = 'expert'

In [ ]:
# Initialize dictionaries to store datasets and environments
datasets = {}
environments = {}

## Training Parameters
Set global hyperparameters used across all experiments.

In [ ]:
n_steps = 10_000               # Total number of training steps
n_steps_per_epoch = 200        # Number of steps in one epoch
batch_size = 512               # Batch size for training
seed = 42                      # Random seed for reproducibility
n_evaluation_episodes = 25     # Number of episodes to average over during evaluation


## Output folder

Create a folder to store logs and trained policies.

In [ ]:
print("\nSetting up folders")

# Create base folder for offline policies
policies_path = os.path.join("policies", "offline")
#models_path = os.path.join("models", "offline")  # (optional) folder for model checkpoints

# Create 'policies/offline' if it doesn't exist
if not os.path.exists(policies_path):
    os.makedirs(policies_path)
    print(f"Created: {policies_path}")
else:
    print(f"Already exists: {policies_path}")

# Create 'models/offline' if needed
#if not os.path.exists(models_path):
#    os.makedirs(models_path)
#    print(f"Created: {models_path}")
#else:
#    print(f"Already exists: {models_path}")

# Create one log folder per task under 'training_logs/offline'
training_base = os.path.join("training_logs", "offline")

for task in tasks:
    task_path = os.path.join(training_base, task)
    if not os.path.exists(task_path):
        os.makedirs(task_path)
        print(f"Created: {task_path}")
    else:
        print(f"Already exists: {task_path}")

## Load Datasets and Environments
We load the offline datasets stored in `.npz` format and recover the corresponding simulation environments using Minari.

In [ ]:
for task in tasks:
    # Load environment from Minari dataset
    environments[task] = minari.load_dataset(f"D4RL/{task}/{dataset_type}-v2").recover_environment()
    
    # Load preprocessed dataset from .npz file
    data = np.load(f"datasets/{task}.npz")
    
    # Create a d3rlpy MDPDataset for training
    datasets[task] = d3rlpy.datasets.MDPDataset(
        observations=data['observations'],
        actions=data['actions'],
        rewards=data['rewards'],
        terminals=data['terminals'],
        action_space=d3rlpy.constants.ActionSpace.CONTINUOUS
    )

## Evaluation Function
Define a custom evaluation protocol that runs multiple episodes in the environment and computes average return.

In [ ]:
class AveragingEvaluator(d3rlpy.metrics.EvaluatorProtocol):
    def __init__(self, env, n_trials=10, alpha=0.0):
        self.env = env
        self.n_trials = n_trials
        # alpha is unused for now, kept for compatibility

    def __call__(self, algo, dataset) -> float:
        rewards = []
        for _ in range(self.n_trials):
            obs, _ = self.env.reset()
            done = False
            total_reward = 0.0

            # Rollout one episode using the policy
            while not done:
                obs_batch = np.expand_dims(obs, axis=0)
                action = algo.predict(obs_batch)[0]

                result = self.env.step(action)
                if len(result) == 5:
                    # Gymnasium-style step (with terminated and truncated)
                    obs, reward, terminated, truncated, _ = result
                    done = terminated or truncated
                else:
                    # Legacy Gym step
                    obs, reward, done, _ = result

                total_reward += reward

            rewards.append(total_reward)

        # Return average reward over all trials
        return float(sum(rewards) / len(rewards))

## Training Function
Define the core training procedure, including algorithm configuration.

In [ ]:
def train_offline_algorithm(config_class, dataset, env, filename, task, algorithm):
    config = config_class()
    
    # Basic training configuration
    config.batch_size = batch_size
    
    # Normalization/scaling is important for stable training
    config.observation_scaler = d3rlpy.preprocessing.StandardObservationScaler()
    config.action_scaler = d3rlpy.preprocessing.MinMaxActionScaler()
    
    # Algorithm-specific hyperparameters
    if config_class == d3rlpy.algos.IQLConfig:
        config.expectile = 0.7
        config.weight_temp = 7.0  
        config.actor_learning_rate = 2e-5 
        config.critic_learning_rate = 2e-5
        
    elif config_class == d3rlpy.algos.TD3PlusBCConfig:
        config.alpha = 0.25
        config.actor_learning_rate = 2e-4
        config.critic_learning_rate = 2e-4
        
    elif config_class == d3rlpy.algos.AWACConfig:
        config.lam = 4
        config.actor_learning_rate = 1e-5
        config.critic_learning_rate = 1e-5
        config.temp_learning_rate = 1e-4
        config.awac_temperature = True
        
    elif config_class == d3rlpy.algos.CQLConfig:
        config.conservative_weight = 10.0
        config.actor_learning_rate = 1e-4
        config.critic_learning_rate = 2e-4
        config.n_action_samples = 50
        
    elif config_class == d3rlpy.algos.BCConfig:
        config.learning_rate = 5e-5

    # Create algorithm instance
    algo = config.create(device="cuda:0" if torch.cuda.is_available() else "cpu")
    algo.build_with_dataset(dataset)
    
    # Set up logging directory
    logger = d3rlpy.logging.FileAdapterFactory(root_dir=f"training_logs/offline/{task}")

    start_time = datetime.now()
    
    # Train the policy offline
    algo.fit(
        dataset,
        n_steps=n_steps,
        n_steps_per_epoch=n_steps_per_epoch,
        evaluators={"evaluation": AveragingEvaluator(env, n_trials=n_evaluation_episodes)},
        logger_adapter=logger,
        show_progress=False,
        save_interval=n_steps_per_epoch
    )

    end_time = datetime.now()
    
    # Save trained policy (model saving for fine-tuning is optional)
    #algo.save_model(f'models/offline/{filename}_model.d3')
    algo.save(f'policies/offline/{filename}_policy.d3')

    # Log summary information
    save_training_summary(filename, algorithm, task, start_time, end_time, config)
    
    print(f"Completed {filename}")
        
    # Free up GPU memory after training
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Save Training Summary
Save key metrics and hyperparameters used during training to a text file for later inspection.

In [ ]:
def save_training_summary(filename, algorithm, task, start_time, end_time, config):
    # Define path to save the summary log
    summary_path = os.path.join("training_logs", "offline", task, f"{filename}_summary.txt")
    total_time = end_time - start_time

    with open(summary_path, "w") as f:
        # General training info
        f.write(f"Task: {task}\n")
        f.write(f"Algorithm: {algorithm}\n")
        f.write(f"Start time: {start_time}\n")
        f.write(f"End time: {end_time}\n")
        f.write(f"Total training time: {total_time}\n")
        f.write(f"n_steps: {n_steps}\n")
        f.write(f"n_steps_per_epoch: {n_steps_per_epoch}\n")
        f.write(f"batch_size: {batch_size}\n")
        f.write(f"seed: {seed}\n")
        f.write(f"n_evaluation_episodes: {n_evaluation_episodes}\n")

        # Dump algorithm-specific hyperparameters
        f.write("\nAlgorithm hyperparameters:\n")
        for key, value in config.__dict__.items():
            if not key.startswith("_"):  # Skip internal/private attributes
                f.write(f"{key}: {value}\n")

## Algorithm Mapping
Map algorithm names to their corresponding configuration classes in `d3rlpy`.

In [ ]:
# Mapping from algorithm name to its corresponding config class
algorithms_config = {     
    'iql': d3rlpy.algos.IQLConfig,  
    'cql': d3rlpy.algos.CQLConfig,  
    'td3bc': d3rlpy.algos.TD3PlusBCConfig,
    'awac': d3rlpy.algos.AWACConfig,    
    'bc': d3rlpy.algos.BCConfig,        
}

## Execute Training
Train the selected algorithms on each task and save the resulting policies and logs.

In [ ]:
print("\nStarting training")
start_time = datetime.now()

# Loop over all tasks and algorithms
for task in tasks:
    for algorithm in algorithms:
        print(f"\nTraining {task} {algorithm}")
        
        # Launch training for the given task/algorithm pair
        train_offline_algorithm(
            algorithms_config[algorithm],
            datasets[task],
            environments[task],
            f"{task}_{algorithm}",  # used for naming saved files
            task,
            algorithm
        )
    
    # Free memory between tasks
    gc.collect()

end_time = datetime.now()
total_time = end_time - start_time

print(f"\nTraining completed in {total_time}")